In [1]:
#!/usr/bin/env python3
"""
🚀 OCCAM Comprehensive Analysis - All Parameters Exposed
This script provides complete control over OCCAM's reconstructability analysis.
"""

# ============================================================================
# 🎛️ COMPLETE OCCAM CONFIGURATION - ALL EXPOSED PARAMETERS
# ============================================================================

# 📁 DATA CONFIGURATION
#DATA_FILE = "SY_sample_pts_to_occam3_shuffle_split_hdr_no_test_sqz.txt"  # Path to your data file
DATA_FILE = "dementia05.txt"  # Path to your data file
OUTPUT_DIR = "occam_results"       # Directory for output files

# 🔍 SEARCH ALGORITHM CONFIGURATION
SEARCH_ALGORITHM = "full-up"       # Options: "loopless-up", "full-up", "disjoint-up", "chain-up"
SEARCH_DIRECTION = "up"            # Options: "up", "down"
SEARCH_LEVELS = 7                  # Lattice depth (1-10+, higher = more thorough)
SEARCH_WIDTH = 3                   # Beam width (3-20+, higher = more thorough)
INCLUDE_TEST_DATA = False          # Include test data columns in analysis

# 🎯 MODEL SELECTION CRITERIA
BEST_MODEL_CRITERIA = "bic"        # Options: "bic", "aic", "information", "alpha"
ALPHA_THRESHOLD = 0.05             # Significance threshold for alpha-based selection
MODEL_SELECTION_MODE = "highest"   # Options: "highest", "lowest" (for the criteria)

# 📊 REFERENCE MODEL CONFIGURATION
REFERENCE_MODEL = "bottom"         # Options: "bottom", "top", custom model name
SORT_ATTRIBUTE = "information"     # Options: "information", "bic", "aic", "alpha", "lr"
SORT_DIRECTION = "descending"      # Options: "ascending", "descending"

# 💾 OUTPUT CONFIGURATION
SAVE_SEARCH_REPORT = True          # Save detailed search report
SAVE_FIT_REPORT = True            # Save model fit report
VERBOSE_OUTPUT = True              # Enable detailed console output

print("🎛️ OCCAM Configuration Loaded")
print("=" * 50)
print(f"📁 Data File: {DATA_FILE}")
print(f"🔍 Search: {SEARCH_ALGORITHM} ({SEARCH_DIRECTION}, L{SEARCH_LEVELS}, W{SEARCH_WIDTH})")
print(f"🎯 Selection: Best by {BEST_MODEL_CRITERIA.upper()}")
print(f"📊 Reference: {REFERENCE_MODEL}")
print(f"💾 Output: {OUTPUT_DIR}/")
print("=" * 50)

# 📦 IMPORTS
import sys
import os
import time
from datetime import datetime
from pathlib import Path

# Create output directory
Path(OUTPUT_DIR).mkdir(exist_ok=True)

try:
    import pyoccam
    print("✅ pyoccam imported successfully")
except ImportError as e:
    print(f"❌ Error importing pyoccam: {e}")
    print("Please ensure pyoccam is properly installed")
    sys.exit(1)

def main():
    """Main analysis function"""

    # 🚀 OCCAM MANAGER INITIALIZATION
    print("\n🔧 Initializing OCCAM Manager...")
    print("-" * 30)

    manager = pyoccam.VBMManager()

    # Construct command line arguments
    cmd_args = ["occam", DATA_FILE]
    if INCLUDE_TEST_DATA:
        cmd_args.append("--test")

    print(f"Command line args: {cmd_args}")

    # Initialize manager
    if not manager.init_from_command_line(cmd_args):
        print("❌ Failed to initialize OCCAM manager")
        return False

    print("✅ OCCAM Manager initialized successfully")

    # 📊 DATASET ANALYSIS
    print("\n📈 Dataset Information:")
    print("-" * 25)

    # Get basic statistics
    stats = manager.get_basic_statistics()
    if 'error' not in stats:
        print(f"Variables: {stats.get('num_variables', 'Unknown')}")
        print(f"Cases: {stats.get('num_cases', 'Unknown')}")
    else:
        print(f"Stats: {stats}")

    # Get variable list
    variables = manager.get_variable_list()
    print(f"\nVariables detected:")
    for i, var in enumerate(variables, 1):
        print(f"   {i}. {var}")

    # 🎯 COMPREHENSIVE SEARCH EXECUTION
    print(f"\n⏳ Performing {SEARCH_ALGORITHM} search...")
    print("-" * 40)
    print(f"🎛️ Search Configuration:")
    print(f"   • Algorithm: {SEARCH_ALGORITHM}")
    print(f"   • Levels: {SEARCH_LEVELS}")
    print(f"   • Width: {SEARCH_WIDTH}")
    print(f"   • Selection Criteria: {BEST_MODEL_CRITERIA.upper()}")

    # Start timing
    search_start_time = time.time()

    # Execute search
    try:
        search_report = manager.generate_search_report(
            SEARCH_ALGORITHM,
            SEARCH_LEVELS,
            SEARCH_WIDTH,
            INCLUDE_TEST_DATA
        )
        search_time = time.time() - search_start_time
        print(f"✅ Search completed in {search_time:.2f} seconds")

    except Exception as e:
        print(f"❌ Search failed: {e}")
        return False

    # 📋 SEARCH RESULTS DISPLAY
    print(f"\n📋 COMPLETE SEARCH REPORT")
    print("=" * 80)
    print(search_report)

    # Save search report
    if SAVE_SEARCH_REPORT:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        search_filename = f"{OUTPUT_DIR}/occam_search_{SEARCH_ALGORITHM}_{timestamp}.txt"

        with open(search_filename, 'w') as f:
            f.write(f"OCCAM Search Report - {SEARCH_ALGORITHM.upper()}\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Data File: {DATA_FILE}\n")
            f.write("=" * 80 + "\n\n")
            f.write(search_report)

        print(f"💾 Search report saved to: {search_filename}")

    # 🎯 INTELLIGENT BEST MODEL EXTRACTION
    print(f"\n🔍 Extracting Best Model by {BEST_MODEL_CRITERIA.upper()}...")
    print("-" * 50)

    # Parse search results to find best model
    lines = search_report.split('\n')
    best_model_name = None

    # Look for the "Best Model(s) by" section
    for i, line in enumerate(lines):
        if f"Best Model(s) by d{BEST_MODEL_CRITERIA.upper()}:" in line:
            if i + 1 < len(lines):
                best_model_name = lines[i + 1].strip()
                break

    # Alternative: Extract from table if marked with asterisk
    if not best_model_name:
        data_lines = [line for line in lines if '\t' in line and not line.startswith('ID')]
        for line in data_lines:
            if '*' in line:
                parts = line.split('\t')
                if len(parts) > 1:
                    best_model_name = parts[1].replace('*', '').strip()
                    break

    if best_model_name:
        print(f"🎯 Best Model Found: {best_model_name}")
        print(f"📊 Selection Criteria: {BEST_MODEL_CRITERIA.upper()}")
    else:
        print("⚠️ Could not automatically extract best model")
        return False

    # 🔬 COMPREHENSIVE MODEL FIT ANALYSIS
    print(f"\n🔬 Performing Detailed Fit Analysis...")
    print("-" * 40)
    print(f"🎯 Target Model: {best_model_name}")

    fit_start_time = time.time()

    try:
        # Generate comprehensive fit report
        fit_report = manager.generate_fit_report(best_model_name)
        fit_time = time.time() - fit_start_time

        print(f"✅ Fit analysis completed in {fit_time:.3f} seconds")

        print(f"\n📊 DETAILED FIT REPORT FOR: {best_model_name}")
        print("=" * 60)
        print(fit_report)

        # Save fit report
        if SAVE_FIT_REPORT:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            fit_filename = f"{OUTPUT_DIR}/occam_fit_{best_model_name.replace(':', '_')}_{timestamp}.txt"

            with open(fit_filename, 'w') as f:
                f.write(f"OCCAM Model Fit Report\n")
                f.write(f"Model: {best_model_name}\n")
                f.write(f"Selection Criteria: {BEST_MODEL_CRITERIA.upper()}\n")
                f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write("=" * 60 + "\n\n")
                f.write(fit_report)

            print(f"💾 Fit report saved to: {fit_filename}")

    except Exception as e:
        print(f"❌ Fit analysis failed: {e}")
        return False

    # 📊 FINAL SUMMARY
    total_time = time.time() - search_start_time
    print(f"\n📊 ANALYSIS COMPLETE!")
    print("=" * 40)
    print(f"🎯 Best Model: {best_model_name}")
    print(f"⚡ Total Time: {total_time:.2f} seconds")
    print(f"📁 Results saved to: {OUTPUT_DIR}/")

    return True

if __name__ == "__main__":
    success = main()
    if success:
        print("\n🎉 OCCAM Analysis completed successfully!")
    else:
        print("\n❌ OCCAM Analysis failed!")
        sys.exit(1)

🎛️ OCCAM Configuration Loaded
📁 Data File: dementia05.txt
🔍 Search: full-up (up, L7, W3)
🎯 Selection: Best by BIC
📊 Reference: bottom
💾 Output: occam_results/
✅ pyoccam imported successfully

🔧 Initializing OCCAM Manager...
------------------------------
Command line args: ['occam', 'dementia05.txt']
✅ OCCAM Manager initialized successfully

📈 Dataset Information:
-------------------------


AttributeError: 'str' object has no attribute 'get'